In [1]:
import os
import json
import shutil
import sys

import numpy as np
import scipy

In [2]:
sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
import topnum

from topnum.scores.perplexity_score import PerplexityScore
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.model_constructor import init_model_from_family, KnownModel, PARAMS_EXPLORED, init_plsa

In [4]:
import artm
from artm import ARTM, Dictionary

import topicnet
from topicnet.cooking_machine.dataset import Dataset
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)

from topicnet.cooking_machine.models.topic_model import ARTM_NINE
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.viewers.top_documents_viewer import TopDocumentsViewer
from topicnet.viewers.top_tokens_viewer import TopTokensViewer
from topicnet.cooking_machine.model_constructor import (
    add_standard_scores,
    create_default_topics,
    count_vocab_size,
    init_model,
)
from topicnet.cooking_machine.rel_toolbox_lite import (
    count_vocab_size,
    modality_weight_rel2abs,
    transform_regularizer,
)


import numpy as np
import pandas as pd
from pandas import DataFrame
from scipy.spatial.distance import cdist

import os
import tempfile
import warnings
from copy import deepcopy
from typing import Dict, List, Optional

In [6]:
NUM_TOPICS = 50  # TODO: this value does not affect anything

NUM_ITERATIONS = 5
NUM_TOP_TOKENS = 20
NUM_TRAINS = 20

In [7]:
BERTOPIC_FOLDER_PATH = '/data_mil/shared/CompressaAI/BERTopic'

In [8]:
! ls $BERTOPIC_FOLDER_PATH

results  results50


In [9]:
! ls $BERTOPIC_FOLDER_PATH/results

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


In [10]:
RESULTS_FOLDER_PATH = os.path.join(BERTOPIC_FOLDER_PATH, 'results50', 'postnauka')

In [11]:
RESULTS_FOLDER_PATH

'/data_mil/shared/CompressaAI/BERTopic/results50/postnauka'

In [12]:
! ls $RESULTS_FOLDER_PATH

0  1  10  11  12  13  14  15  16  17  18  19  2  3  4  5  6  7	8  9


In [13]:
! ls $RESULTS_FOLDER_PATH/0

dataset.csv  phi.csv  top_words.json


In [14]:
dataset = Dataset(
    f'{RESULTS_FOLDER_PATH}/0/dataset.csv',
)

dataset.get_possible_modalities()

{'@word'}

In [15]:
MAIN_MODALITY = '@word'

In [16]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        _T, W = phi.shape
        T = len(topic_indices)

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [17]:
%%time

occurences, co_occurences = calc_doc_occurrences(dataset, MAIN_MODALITY)

CPU times: user 10.9 s, sys: 1.77 s, total: 12.7 s
Wall time: 12.6 s


In [18]:
co_occurences.shape

(82113, 82113)

In [19]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    dataset.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [20]:
class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

In [21]:
def view_model(
        topic_model,
        dataset,
        num_top_tokens: int = 5,
        top_tokens_method: str = 'phi',
        num_topics: Optional[int] = 5,  # we do not want to fill the whole .ipynb notebook with topics...
        ):
    top_tok_viewer = TopTokensViewer(
        topic_model, num_top_tokens=num_top_tokens, method=top_tokens_method
    )
    top_doc_viewer = TopDocumentsViewer(topic_model, dataset=dataset)
    top_docs = top_doc_viewer.view()

    if num_topics is None:
        num_topics = len(topic_model.topic_names)

    for topic_name in topic_model.topic_names[:num_topics]:
        topic_top_toks = top_tok_viewer.to_html(topic_names=[topic_name])
        topic_top_docs = top_docs[topic_name]
        display_html(topic_top_toks, raw=True)
        display(topic_top_docs)

In [22]:
class FastFixPhiRegularizer(BaseRegularizer):
    _VERY_BIG_TAU = 10 ** 9

    def __init__(self, name: str, topic_names: List[str], parent_model=None, parent_phi=None, words=None):
        super().__init__(name, tau=self._VERY_BIG_TAU)

        self._topic_names = topic_names
        self._topic_indices = None
        self._words = words
        self._word_indices = None
        self._parent_model = parent_model
        self._parent_phi = parent_phi

    def grad(self, pwt, nwt):
        # print('Fixing')

        rwt = np.zeros_like(pwt)

        if self._parent_phi is not None:
            parent_phi = self._parent_phi
            vals = parent_phi.values
        else:
            assert False

            parent_phi = self._parent_model.get_phi()
            vals = parent_phi.values[:, self._topic_indices]

        if self._word_indices is None:
            assert vals.shape[0] == rwt.shape[0], (vals.shape[0], rwt.shape[0])
        else:
            assert vals.shape[0] == len(self._word_indices)

        assert vals.shape[1] == len(self._topic_indices)

        if self._word_indices is None:
            rwt[:, self._topic_indices] += vals
        else:
            # print(len(self._word_indices), len(self._topic_indices), rwt.shape, vals.shape)
            # print(self._word_indices[:3], self._topic_indices[:3])
            # print(rwt[np.ix_(self._word_indices, self._topic_indices)].shape, vals.shape)

            # https://github.com/numpy/numpy/issues/5574
            # https://github.com/numpy/numpy/issues/13255
            rwt[np.ix_(self._word_indices, self._topic_indices)] += vals

            # print(np.ix_(self._word_indices, self._topic_indices)[:3])

        return self.tau * rwt

    def attach(self, model):
        super().attach(model)
        
        phi = self._model.get_phi()
        self._topic_indices = [
            phi.columns.get_loc(topic_name)
            for topic_name in self._topic_names
        ]

        if self._words is not None:
            self._word_indices = [
                # phi.index.get_loc(w) for w in self._words
                int(phi.index.get_loc(w)) for w in self._words
            ]

            # print(f'!!! Word indices: {self._word_indices}')

In [23]:
def fit_and_compute_scores(model, dataset, target_topic_indices=None, custom_regularizers=None):
    print(custom_regularizers)

    model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS, custom_regularizers=custom_regularizers)

    score_values = {
        'perplexity': model.scores[f'PerplexityScore{MAIN_MODALITY}'][-1],
    }

    phi = model.get_phi()

    # Currently all topics are taken into account

    if target_topic_indices is None:
        target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()

    target_topic_names = [phi.columns[i] for i in target_topic_indices]

    top = NUM_TOP_TOKENS
    coherence_score = TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )

    value = coherence_score.call(model)
    score_values[coherence_score._name] = value
    topic_coherences = coherence_score.call_by_topic(model)
    topic_coherences = {t: float(v) for t, v in topic_coherences.items()}

    diversity_scores = [
        DiversityScore(
            name=f'diversity_{metric}',
            metric=metric,
            topic_names=target_topic_names,
            class_ids=MAIN_MODALITY,
        )
    
        for metric in KNOWN_METRICS
    ]
    
    for score in diversity_scores:
        value = score.call(model)
        score_values[score._name] = value

    return {
        'scores': score_values,
        'topic_coherences': topic_coherences,
    }

In [24]:
def init_model_from_family(
        family: str or KnownModel,
        dataset: Dataset,
        main_modality: str,
        num_topics: int,
        seed: int,
        specific_topic_names = None,
        modalities_to_use: List[str] = None,
        num_processors: int = 3,
        model_params: dict = None,
):
    """
    Returns
    -------
    model: TopicModel() instance
    """
    if isinstance(family, KnownModel):
        family = family.value

    if modalities_to_use is None:
        modalities_to_use = [main_modality]

    custom_regs = {}

    if family == "LDA":
        model = init_lda(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "PLSA":
        model = init_plsa(
            dataset, modalities_to_use, main_modality, num_topics
        )
    elif family == "TARTM":
        model, custom_regs = init_thetaless(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "sparse":
        model = init_bcg_sparse_model(
            dataset, modalities_to_use, main_modality, num_topics, 1, model_params
        )
    elif family == "decorrelation":
        model = init_decorrelated_plsa(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "ARTM":
        model = init_baseline_artm(
            dataset, modalities_to_use, main_modality, num_topics, 1, specific_topic_names, model_params
        )
    else:
        raise ValueError(f'family: {family}')

    model.num_processors = num_processors

    if seed is not None:
        model.seed = seed

    dictionary = dataset.get_dictionary()

    # TODO: maybe this cycle is not necessary
    for modality in dataset.get_possible_modalities():
        if modality not in modalities_to_use:
            dictionary.filter(class_id=modality, max_df=0, inplace=True)

    model.initialize(dictionary)
    add_standard_scores(model, dictionary, main_modality=main_modality,
                        all_modalities=modalities_to_use)

    model = TopicModel(
        artm_model=model,
        custom_regularizers=custom_regs
    )

    return model


def init_bcg_sparse_model(
        dataset,
        modalities_to_use,
        main_modality,
        specific_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str or dict
    main_modality : str
    specific_topics : int
    bcg_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_plsa(
        dataset, modalities_to_use, main_modality, specific_topics, bcg_topics
    )
    background_topic_names = model.topic_names[-bcg_topics:]

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    dictionary = dataset.get_dictionary()
    baseline_class_ids = {class_id: 1 for class_id in modalities_to_use}
    data_stats = count_vocab_size(dictionary, baseline_class_ids)

    # all coefficients are relative
    regularizers = [
        artm.SmoothSparsePhiRegularizer(
             name='smooth_phi_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
             class_ids=[main_modality],
        ),
        artm.SmoothSparseThetaRegularizer(
             name='smooth_theta_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
        ),
        artm.SmoothSparsePhiRegularizer(
             name='sparse_phi_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
             class_ids=[main_modality],
            ),
        artm.SmoothSparseThetaRegularizer(
             name='sparse_theta_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
        ),
    ]
    for reg in regularizers:
        model.regularizers.add(transform_regularizer(
            data_stats,
            reg,
            model.class_ids,
            n_topics=len(reg.topic_names)
        ))

    return model


def init_baseline_artm(
        dataset,
        modalities_to_use,
        main_modality,
        num_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None,
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str
    main_modality : str
    num_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_bcg_sparse_model(
        dataset, modalities_to_use, main_modality, num_topics, bcg_topics, specific_topic_names, model_params
    )

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    model.regularizers.add(
        artm.DecorrelatorPhiRegularizer(
            gamma=0,
            tau=model_params.get('decorrelation_tau', 0.01),
            name='decorrelation',
            topic_names=specific_topic_names,
            class_ids=modalities_to_use,
        )
    )

    return model

In [25]:
TOPIC_INDICES = list(range(NUM_TOPICS))

In [26]:
TOPIC_INDICES

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49]

In [27]:
phi0 = pd.read_csv(f'{RESULTS_FOLDER_PATH}/0/phi.csv', index_col=0)

In [28]:
phi0.head()

,background_1,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,...,topic_40,topic_41,topic_42,topic_43,topic_44,topic_45,topic_46,topic_47,topic_48,topic_49
aa,0.000044,0.000000,0.000112,0.0,0.00000,0.0,0.0,0.000062,0.0,0.0,...,0.001094,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
aaaaaaaa,0.000000,0.000000,0.000066,0.0,0.00000,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
aacn,0.000000,0.000000,0.000066,0.0,0.00000,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
aar,0.000000,0.000068,0.000000,0.0,0.00000,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
aarhus,0.000000,0.000000,0.000000,0.0,0.00007,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [29]:
phi0.set_index([[MAIN_MODALITY] * len(phi0.index), phi0.index], inplace=True)

In [30]:
phi0.head()

background_1   topic_0   topic_1  topic_2  topic_3  topic_4  \
@word aa            0.000044  0.000000  0.000112      0.0  0.00000      0.0   
      aaaaaaaa      0.000000  0.000000  0.000066      0.0  0.00000      0.0   
      aacn          0.000000  0.000000  0.000066      0.0  0.00000      0.0   
      aar           0.000000  0.000068  0.000000      0.0  0.00000      0.0   
      aarhus        0.000000  0.000000  0.000000      0.0  0.00007      0.0   

                topic_5   topic_6  topic_7  topic_8  ...  topic_40  topic_41  \
@word aa            0.0  0.000062      0.0      0.0  ...  0.001094       0.0   
      aaaaaaaa      0.0  0.000000      0.0      0.0  ...  0.000000       0.0   
      aacn          0.0  0.000000      0.0      0.0  ...  0.000000       0.0   
      aar           0.0  0.000000      0.0      0.0  ...  0.000000       0.0   
      aarhus        0.0  0.000000      0.0      0.0  ...  0.000000       0.0   

                topic_42  topic_43  topic_44  topic_45  topic_46  topic_47  \
@word aa             0.0       0.0       0.0       0.0       0.0       0.0   
      aaaaaaaa       0.0       0.0       0.0       0.0       0.0       0.0   
      aacn           0.0       0.0       0.0       0.0       0.0       0.0   
      aar            0.0       0.0       0.0       0.0       0.0       0.0   
      aarhus         0.0       0.0       0.0       0.0       0.0       0.0   

                topic_48  topic_49  
@word aa             0.0       0.0  
      aaaaaaaa       0.0       0.0  
      aacn           0.0       0.0  
      aar            0.0       0.0  
      aarhus         0.0       0.0  

[5 rows x 51 columns]

In [31]:
DIFF_THRESHOLD = 2

In [32]:
def check_top_words(phi, top_words):
    diffs = []

    for t, topic_top_words in top_words.items():
        if t == 'background_1':
            print(f'Skipping background topic: {t}.')

            continue

        print(t)
        # print(top_words)
    
        top_phi = set(phi[t].sort_values(ascending=False)[:NUM_TOP_TOKENS].index.get_level_values(1))
        top_bt = set([p[0] for p in topic_top_words])
    
        if top_phi == top_bt:
            diffs.append(
                {
                    'total': 0,
                    'lost_bt': 0,
                    'lost_model': 0,
                }
            )
        else:
            diff1 = top_phi.difference(top_bt)
            diff2 = top_bt.difference(top_phi)
    
            print('  WTF:', diff1, diff2)
    
            if len(diff1) > DIFF_THRESHOLD:
                print(f'  WTF?!?!?', len(diff1))
    
            if len(diff2) > DIFF_THRESHOLD:
                print(f'  WTF?!?!?', len(diff2))

            diffs.append(
                {
                    'total': len(diff1 | diff2),
                    'lost_bt': len(diff2),
                    'lost_model': len(diff1),
                }
            )

    return diffs

In [33]:
def init_model_and_phi(num_topics, dataset, phi0):
    model = init_model_from_family(
        family=KnownModel.PLSA,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=num_topics,
        seed=0,
    )
    
    phi = model.get_phi()
    
    # assert phi.shape[1] == phi0.shape[1] - 1
    assert phi.shape[1] == num_topics
    assert phi.shape[1] in [phi0.shape[1], phi0.shape[1] - 1]
    
    common_words = list(set(phi.index).intersection(phi0.index))
    
    phi.loc[:, :] = 0

    if phi.shape[1] == phi0.shape[1]:
        target_topics = phi0.columns
    elif phi.shape[1] == phi0.shape[1] - 1:
        target_topics = phi.columns
    else:
        assert False

    # print(common_words, phi.index, phi0.index)

    phi.loc[common_words, :] += phi0.loc[common_words, target_topics]

    diff = set(phi0.columns).symmetric_difference(set(phi.columns))

    assert diff == {'background_1'} or diff == {'background_1', f'topic_{len(phi.columns) - 1}'}
    
    phi = phi / phi.sum(axis=0)

    return model, phi, common_words

In [34]:
! ls results50

20newsgroups  mkb10  postnauka	rtlwikiperson  ruwikigood


In [36]:
SAVE_FOLDER = os.path.join('results50', 'postnauka')

In [38]:
SAVE_FOLDER

'results50/postnauka'

In [39]:
! ls $SAVE_FOLDER

ablation_study	    iterative_100000.json      lda.json     tless.json
decorrelation.json  iterative2_100000000       plsa.json
iterative_100000    iterative2_100000000.json  sparse.json


In [40]:
results = []

is_results_loaded = False

save_file_path = os.path.join(
    SAVE_FOLDER, 'bertopic.json'
)

if os.path.isfile(save_file_path):
    results = json.loads(
        open(save_file_path).read()
    )
    is_results_loaded = True


singles_save_folder = os.path.join(
    SAVE_FOLDER, 'bertopic'
)

os.makedirs(singles_save_folder, exist_ok=True)


for seed in range(NUM_TRAINS):
    if is_results_loaded:
        continue

    print(seed)

    current_save_file_path = os.path.join(
        singles_save_folder, f'bertopic_{seed}.json'
    )

    if os.path.isfile(current_save_file_path):
        print(f'Already computed results: {current_save_file_path}. Loading and skipping...')

        result = json.loads(
            open(current_save_file_path).read()
        )
        results.append(result)

        continue

    seed_load_folder = os.path.join(RESULTS_FOLDER_PATH, str(seed))
    files = [f for f in os.listdir(seed_load_folder) if os.path.isfile(f'{seed_load_folder}/{f}')]
    
    assert len(files) == 3

    dataset = Dataset(
        f'{seed_load_folder}/dataset.csv',
    )
    phi0 = pd.read_csv(f'{seed_load_folder}/phi.csv', index_col=0)
    phi0.set_index([[MAIN_MODALITY] * len(phi0.index), phi0.index], inplace=True)

    assert all('back' not in t for t in phi0.columns[1:])
    assert 'back' in phi0.columns[0]

    num_specific_topics = phi0.shape[1] - 1

    assert num_specific_topics == len([t for t in phi0.columns if 'back' not in t])

    print(f'Num model topics: {phi0.shape[1]}.')

    with open(f'{seed_load_folder}/top_words.json', 'r') as f:
        top_words = json.loads(f.read())


    
    model, phi, common_phi_words = init_model_and_phi(
        num_topics=num_specific_topics + 1,  # background
        dataset=dataset, phi0=phi0
    )
    common_words = [
        w[1] for w in common_phi_words  # without modality
    ]
    common_words_phi0_indices = [
        # phi0.index.get_loc(w) for w in common_words
        int(phi0.index.get_locs(w)) for w in common_phi_words
    ]

    print('Check before fit:')
    diff_tops1 = check_top_words(phi, top_words)  # Whatever...
    
    fix_regularizer = FastFixPhiRegularizer(
        name='fix',
        parent_phi=phi0.iloc[common_words_phi0_indices, 1:],
        words=common_words,
        topic_names=phi.columns[1:],
    )
    result = fit_and_compute_scores(
        model, dataset,
        target_topic_indices=list(range(phi.shape[1])),
        custom_regularizers = {
            fix_regularizer.name: fix_regularizer,
        }
    )

    print('Check after fit:')
    diff_tops2 = check_top_words(phi, top_words)  # Whatever...

    assert diff_tops1 == diff_tops2

    fair_ppl_free = result['scores']['perplexity']

    del model, phi, fix_regularizer, result


    
    model, phi, _ = init_model_and_phi(
        num_topics=num_specific_topics + 1,  # background
        dataset=dataset, phi0=phi0
    )
    fix_regularizer = FastFixPhiRegularizer(
        name='fix',
        parent_phi=phi0.iloc[common_words_phi0_indices, :],  # Diff here
        words=common_words,
        topic_names=phi.columns,
    )
    result = fit_and_compute_scores(
        model, dataset,
        target_topic_indices=list(range(phi.shape[1])),
        custom_regularizers = {
            fix_regularizer.name: fix_regularizer,
        }
    )
    
    fair_ppl_fix = result['scores']['perplexity']

    del model, phi, fix_regularizer, result

    
    
    model, phi, _ = init_model_and_phi(
        num_topics=num_specific_topics,  # Diff here
        dataset=dataset, phi0=phi0
    )
    fix_regularizer = FastFixPhiRegularizer(
        name='fix',
        parent_phi=phi0.iloc[common_words_phi0_indices, 1:],  # Diff here
        words=common_words,
        topic_names=phi.columns,
    )
    result = fit_and_compute_scores(
        model, dataset,
        custom_regularizers = {
            fix_regularizer.name: fix_regularizer,
        }
    )

    unfair_ppl_banklike = result['scores']['perplexity']


    
    result['scores']['fair_ppl_free'] = fair_ppl_free
    result['scores']['fair_ppl_fix'] = fair_ppl_fix
    result['scores']['unfair_ppl_banklike'] = unfair_ppl_banklike

    assert result['scores']['fair_ppl_free'] < result['scores']['unfair_ppl_banklike']
    # assert result['scores']['fair_ppl_fix'] < result['scores']['unfair_ppl_banklike']

    result['scores']['coherence_20'] = float(result['scores']['coherence_20'])
    result['stats'] = {
        'num_topics': phi0.shape[1],
        'num_common_words': len(common_words),
        'num_model_words': phi.shape[0],
        'num_bt_words': phi0.shape[0],
        'top_diffs': diff_tops2,
    }

    results.append(result)

    print(result['scores'])
    print(result['stats'])

    dumped_result = json.dumps(
        results[-1], indent=4
    )

    with open(current_save_file_path, 'w') as f:
        f.write(dumped_result)

    del model, phi, fix_regularizer


with open(save_file_path, 'w') as f:
    f.write(
        json.dumps(
            results, indent=4
        )
    )

0
Num model topics: 51.


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



Check before fit:
Skipping background topic: background_1.
topic_0
topic_1
topic_2
topic_3
topic_4
topic_5
topic_6
topic_7
topic_8
topic_9
topic_10
topic_11
topic_12
topic_13
topic_14
topic_15
topic_16
topic_17
topic_18
topic_19
topic_20
topic_21
topic_22
topic_23
topic_24
topic_25
topic_26
topic_27
topic_28
topic_29
topic_30
topic_31
topic_32
topic_33
topic_34
topic_35
topic_36
topic_37
topic_38
topic_39
topic_40
topic_41
topic_42
topic_43
topic_44
topic_45
topic_46
  WTF: {'дуэт'} {'филин'}
topic_47
topic_48
topic_49
{'fix': <__main__.FastFixPhiRegularizer object at 0x7f7c8e5b5f10>}
Check after fit:
Skipping background topic: background_1.
topic_0
topic_1
topic_2
topic_3
topic_4
topic_5
topic_6
topic_7
topic_8
topic_9
topic_10
topic_11
topic_12
topic_13
topic_14
topic_15
topic_16
topic_17
topic_18
topic_19
topic_20
topic_21
topic_22
topic_23
topic_24
topic_25
topic_26
topic_27
topic_28
topic_29
topic_30
topic_31
topic_32
topic_33
topic_34
topic_35
topic_36
topic_37
topic_38
topic_39


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f7c9648aac0>}


...

Check before fit:
Skipping background topic: background_1.
topic_0
topic_1
topic_2
topic_3
topic_4
topic_5
topic_6
topic_7
topic_8
topic_9
topic_10
topic_11
topic_12
topic_13
topic_14
topic_15
topic_16
topic_17
topic_18
topic_19
topic_20
topic_21
topic_22
topic_23
topic_24
topic_25
topic_26
topic_27
topic_28
topic_29
topic_30
topic_31
topic_32
topic_33
topic_34
topic_35
topic_36
topic_37
topic_38
topic_39
topic_40
topic_41
topic_42
topic_43
topic_44
topic_45
topic_46
topic_47
topic_48
topic_49
{'fix': <__main__.FastFixPhiRegularizer object at 0x7f7ca24c61c0>}
Check after fit:
Skipping background topic: background_1.
topic_0
topic_1
topic_2
topic_3
topic_4
topic_5
topic_6
topic_7
topic_8
topic_9
topic_10
topic_11
topic_12
topic_13
topic_14
topic_15
topic_16
topic_17
topic_18
topic_19
topic_20
topic_21
topic_22
topic_23
topic_24
topic_25
topic_26
topic_27
topic_28
topic_29
topic_30
topic_31
topic_32
topic_33
topic_34
topic_35
topic_36
topic_37
topic_38
topic_39
topic_40
topic_41
topic_42

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f7ca2724640>}


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f7c878b3fa0>}
{'perplexity': 4650.22021484375, 'coherence_20': 1.2556515537915496, 'diversity_euclidean': 0.05034293665187094, 'diversity_jensenshannon': 0.6732702475822255, 'diversity_hellinger': 0.7914879424242489, 'diversity_cosine': 0.7904769611656064, 'fair_ppl_free': 3041.006591796875, 'fair_ppl_fix': 4448.97265625, 'unfair_ppl_banklike': 4650.22021484375}
{'num_topics': 51, 'num_common_words': 82061, 'num_model_words': 82113, 'num_bt_words': 82061, 'top_diffs': [{'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt': 0, 'lost_model': 0}, {'total': 0, 'lost_bt':

In [42]:
1

1